# 衝突率とスループット解析ノート

このノートブックは、PIM と Slotted Aloha のログを読み込み、
衝突率と **1ノードあたりスループット** を比較するためのものです。

## 1. ライブラリの読み込み

解析とプロットに必要なライブラリを読み込みます。

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import japanize_matplotlib
import pandas as pd
import yaml

## 2. PDFの出力先設定

ログの入力元と、図を保存するPDFファイルの出力先をここで指定します。
必要に応じてこのセルだけ変更してください。

In [8]:
# ログパス
PIM_LOG_DIR = Path("logs/pim_20260428_220852")
SLOTTED_ALOHA_LOG_DIR = Path("logs/slotted_aloha_20260428_221136")

# PDF出力先
OUTPUT_DIR = Path("img/analysis")
COLLISION_RATE_PDF = OUTPUT_DIR / "collision_rate_comparison_nodes_1_30_1_update.pdf"
THROUGHPUT_PER_NODE_BPS_PDF = OUTPUT_DIR / "throughput_per_node_bps_comparison_nodes_1_30_1_update.pdf"

## 3. 解析コード

イベントログを読み込んで、ノード数ごとの衝突率と **1ノードあたりスループット** を計算します。

In [9]:
def load_config(config_path: Path) -> dict:
    """設定ファイルを読み込み、解析に必要なパラメータ辞書を返す。"""
    with config_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_events(events_path: Path) -> pd.DataFrame:
    """events_N.tsvを読み込み、時刻とラウンドを数値列へ整形して返す。"""
    df = pd.read_csv(events_path, sep="\t")
    df["time_tu"] = pd.to_numeric(df["time_tu"], errors="coerce")
    df["round"] = pd.to_numeric(df["round"], errors="coerce").astype(int)
    df["event"] = df["event"].astype(str)
    df["result"] = df["result"].astype(str)
    return df


def iter_event_files(log_dir: Path) -> list[tuple[int, Path]]:
    """log_dir配下の events_N.tsv を (N, Path) で返す。"""
    files = []
    for path in log_dir.glob("events_*.tsv"):
        m = re.fullmatch(r"events_(\d+)\.tsv", path.name)
        if m is None:
            continue
        files.append((int(m.group(1)), path))
    return sorted(files, key=lambda x: x[0])


def success_event_mask(protocol: str, events: pd.DataFrame) -> pd.Series:
    """プロトコルごとの成功イベント条件を返す。"""
    if protocol == "pim":
        return events["event"] == "rx_success"
    if protocol == "slotted_aloha":
        return (events["event"] == "data_response") & (events["result"] == "success")
    raise ValueError(f"未知のプロトコルです: {protocol}")


def summarize_protocol(protocol: str, log_dir: Path) -> pd.DataFrame:
    """
    1プロトコル分のログから、nodes別の衝突率とスループット(bps)を算出する。

    - Slotted Aloha の衝突率は、mask_len=0 のスロット割当の試行ごとに、
      1つでも collision イベントがあればその試行を衝突ありとして数える。
    - PIM の衝突率は、全送信試行数に対する collision イベントの割合で算出する。
    """
    config = load_config(log_dir / "config.yaml")
    data_bits = float(config["data_bits"])
    tu_sec = float(config["tu"])
    event_files = iter_event_files(log_dir)

    if not event_files:
        raise FileNotFoundError(f"events_N.tsv が見つかりません: {log_dir}")

    rows = []
    for num_nodes, events_path in event_files:
        events = load_events(events_path)
        if events.empty:
            continue

        success_mask = success_event_mask(protocol, events)
        successes = int(success_mask.sum())

        if protocol == "slotted_aloha":
            total_trials = 0
            trial_collisions = 0
            total_successes_slot = 0

            for rnd, group in events.groupby("round"):
                inv_cmds = group[
                    (group["event"] == "inventory_cmd") &
                    (group["detail"].str.contains("mask_len=0", na=False))
                ]
                if inv_cmds.empty:
                    continue

                total_trials += 1
                start_time = float(inv_cmds.iloc[0]["time_tu"])

                all_inv_cmds = group[group["event"] == "inventory_cmd"]
                next_cmds = all_inv_cmds[all_inv_cmds["time_tu"] > start_time]
                end_time = float(next_cmds["time_tu"].min()) if not next_cmds.empty else None

                slot_block = group[
                    (group["phase"] == "inventory") &
                    (group["time_tu"] >= start_time)
                ]
                if end_time is not None:
                    slot_block = slot_block[slot_block["time_tu"] < end_time]

                if (slot_block["event"] == "collision").any():
                    trial_collisions += 1

                total_successes_slot += int(
                    ((slot_block["event"] == "tag_response") & (slot_block["result"] == "success")).sum()
                )

            collision_rate = (trial_collisions / total_trials) if total_trials > 0 else 0.0
            collision_successes = total_successes_slot
        else:
            collisions = int((events["event"] == "collision").sum())
            total_attempts = int((events["event"] == "tx_preamble").sum())
            collision_rate = (collisions / total_attempts) if total_attempts > 0 else 0.0

        round_last_tu = events.groupby("round")["time_tu"].max()
        num_rounds = int(round_last_tu.shape[0])
        avg_round_tu = float(round_last_tu.mean()) if num_rounds > 0 else 0.0
        avg_round_sec = avg_round_tu * tu_sec
        avg_successes_per_round = (successes / num_rounds) if num_rounds > 0 else 0.0
        print(f"Protocol: {protocol}, Nodes: {num_nodes}, Successes: {successes}, Avg round time (sec): {avg_round_sec:.3f}")
        throughput_bps = (
            (avg_successes_per_round * data_bits / avg_round_sec)
            if avg_round_sec > 0
            else 0.0
        )
        throughput_per_node_bps = (throughput_bps / num_nodes) if num_nodes > 0 else 0.0

        row = {
            "protocol": protocol,
            "nodes": int(num_nodes),
            "successes": successes,
            "collision_rate": collision_rate,
            "throughput_per_node_bps": throughput_per_node_bps,
        }
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["nodes", "protocol"]).reset_index(drop=True)


def plot_collision_rate(df: pd.DataFrame, output_pdf: Path) -> None:
    """nodesに対する衝突率をPIM/Slotted Alohaで比較し、PDFに出力する。"""
    pivot = df.pivot(index="nodes", columns="protocol", values="collision_rate").sort_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    for protocol in pivot.columns:
        ax.plot(pivot.index, pivot[protocol], marker="o", label=protocol)

    ax.set_title("Collision Rate vs Selected Tags")
    ax.set_xlabel("Selected Tags (nodes)")
    ax.set_ylabel("Collision Rate")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def plot_throughput_bps(df: pd.DataFrame, output_pdf: Path) -> None:
    """nodesに対する1Tagあたりのスループット(bps)をPIM/Slotted Alohaで比較し、PDFに出力する。"""
    pivot = df.pivot(index="nodes", columns="protocol", values="throughput_per_node_bps").sort_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    for protocol in pivot.columns:
        ax.plot(pivot.index, pivot[protocol], marker="o", label=protocol)

    ax.set_title("Throughput per Tag (bps) vs Selected Tags")
    ax.set_xlabel("Selected Tags (nodes)")
    ax.set_ylabel("Throughput per Tag (bps)")
    ax.grid(alpha=0.3)
    ax.legend()

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

## 4. 実行

このセルを実行すると、比較解析を行ってPDFを出力し、結果テーブルを表示します。

In [10]:
pim_df = summarize_protocol("pim", PIM_LOG_DIR)
slotted_df = summarize_protocol("slotted_aloha", SLOTTED_ALOHA_LOG_DIR)
merged = pd.concat([pim_df, slotted_df], ignore_index=True)

plot_collision_rate(merged, COLLISION_RATE_PDF)
plot_throughput_bps(merged, THROUGHPUT_PER_NODE_BPS_PDF)

print("比較解析が完了しました。")
print(f"PIM log dir: {PIM_LOG_DIR}")
print(f"Slotted Aloha log dir: {SLOTTED_ALOHA_LOG_DIR}")
print(f"Collision rate PDF: {COLLISION_RATE_PDF}")
print(f"Throughput PDF: {THROUGHPUT_PER_NODE_BPS_PDF}")
print()
print("=== 解析結果 ===")
print(merged[["protocol", "nodes", "collision_rate", "throughput_per_node_bps"]].to_string(index=False))

Protocol: pim, Nodes: 1, Successes: 10000, Avg round time (sec): 0.005
Protocol: pim, Nodes: 5, Successes: 39843, Avg round time (sec): 0.009
Protocol: pim, Nodes: 10, Successes: 59679, Avg round time (sec): 0.009
Protocol: pim, Nodes: 15, Successes: 67723, Avg round time (sec): 0.010
Protocol: pim, Nodes: 20, Successes: 67677, Avg round time (sec): 0.010
Protocol: pim, Nodes: 25, Successes: 63844, Avg round time (sec): 0.010
Protocol: pim, Nodes: 30, Successes: 57634, Avg round time (sec): 0.010
Protocol: slotted_aloha, Nodes: 1, Successes: 10000, Avg round time (sec): 0.028
Protocol: slotted_aloha, Nodes: 5, Successes: 49200, Avg round time (sec): 0.087
Protocol: slotted_aloha, Nodes: 10, Successes: 96636, Avg round time (sec): 0.175
Protocol: slotted_aloha, Nodes: 15, Successes: 142216, Avg round time (sec): 0.265
Protocol: slotted_aloha, Nodes: 20, Successes: 185868, Avg round time (sec): 0.355
Protocol: slotted_aloha, Nodes: 25, Successes: 227572, Avg round time (sec): 0.441
Proto